# Sample and label noise clips

Collect more clips for the noise model's 13 classes:

1. **Choose a class** you're collecting.
2. **Pick sites** on the map. It only shows sites that don't have that class yet.
3. **Sample** that class's detections at those sites, with the same limits as `data/metadata_noise_model.csv`.
4. **Label every class you hear** in each clip.
5. **Check progress**: `data/metadata_noise_model.csv` plus your labels against the targets.

**You need:** a `.env` file in the repo folder with `API_TOKEN` (to find detections) and AWS keys (to play the audio).

**Run it** in JupyterLab (`pixi run -e annotate jupyter`), one step at a time. The annotator in step 4 doesn't work in
VS Code's notebook viewer.

In [1]:
import os
import re
import zlib
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from dotenv import load_dotenv
from jupyter_bioacoustic import BioacousticAnnotator

# Work from the repo folder, so every path here and in annotator_config/ starts there.
if Path.cwd().name == "notebooks":
    os.chdir("..")

CLASSES = ["engine", "fireworks", "gun", "power_tools", "siren", "human_vocal", "human_non_vocal",
           "generator", "traffic", "airplane", "device_static", "wind", "water"]
# Targets and caps per class, the same as data/metadata_noise_model.csv (scripts/noise_model_metadata.py)
TARGETS = {"clips": 200, "sites": 20, "recordings": 40, "negatives": 100}
MAX_PER_RECORDING, MAX_PER_SITE, MAX_NEG_PER_SITE = 5, 10, 5
MIN_GAP_SECONDS = 60  # between clips from one recording, so each clip is a separate event
SEED = 0

DATASET = Path("data/metadata_noise_model.csv")    # clips we already have
CLIPS = Path("data/annotation/noise_clips.csv")    # clips sampled for labeling (step 3 adds to it)
LABELS = Path("data/annotation/noise_labels.csv")  # your labels (step 4 saves them as you go)
CLIP_COLUMNS = ["id", "site", "recording_id", "recording_name", "detected_label", "confidence",
                "start_time", "end_time", "path", "sampled_on"]

load_dotenv(".env", override=True)  # API_TOKEN, and the AWS keys the annotator uses to play audio from S3
API_URL = "https://api.wildlifesoundhub.org"
COOKIES = {"__Secure-authjs.session-token": os.environ["API_TOKEN"]}


def query_api(path, timeout=120, **params):
    """GET `{API_URL}{path}?{params}` with the session cookie and return the JSON body."""
    r = requests.get(f"{API_URL}{path}", params=params, cookies=COOKIES, timeout=timeout)
    r.raise_for_status()
    return r.json()


query_api("/users/me")  # a 401 error here means API_TOKEN in .env has expired
deployments = pd.DataFrame(query_api("/deployments/")).rename(columns={"id": "deployment_id", "placename": "site"})
deployments[["lon", "lat"]] = [g["coordinates"] for g in deployments.geometry]
recordings = pd.DataFrame(query_api("/recordings/")).merge(
    deployments[["deployment_id", "site", "lat", "lon"]], on="deployment_id")
# Start time from the file name (e.g. 20230526_000000), as in notebook 03; NaT if the name has none
recordings["time"] = pd.to_datetime(recordings.path.str.extract(r"(\d{8}_\d{6})")[0],
                                    format="%Y%m%d_%H%M%S", errors="coerce")
recordings["recording_name"] = recordings.site + "/" + recordings.path.map(lambda p: Path(p).stem)
sites = recordings.groupby("site", as_index=False).agg(lat=("lat", "mean"), lon=("lon", "mean"),
                                                         recordings=("id", "size"))
print(f"Connected: {len(sites)} sites, {len(recordings):,} recordings")

Connected: 582 sites, 24,127 recordings


In [2]:
def labeled():
    """Labeled clips that count (not flagged unsure or unusable): clip list rows with a 0/1 column per class."""
    if not LABELS.exists():
        return pd.DataFrame(columns=["site", "recording_name", *CLASSES])
    labels = pd.read_csv(LABELS).drop_duplicates("id", keep="last")
    labels = labels[(labels.unsure != 1) & (labels.unusable != 1)]
    return pd.read_csv(CLIPS).merge(labels[["id", *CLASSES]], on="id")


def class_rows():
    """One row per clip and class: label 1 = present, 0 = absent. Dataset clips count for their own class only."""
    ds = pd.read_csv(DATASET)
    rows = [pd.DataFrame({"cls": ds.common_name, "site": ds.site, "recording": ds.recording_name, "label": 1})]
    lab = labeled()
    if len(lab):
        rows.append(lab.melt(id_vars=["site", "recording_name"], value_vars=CLASSES, var_name="cls",
                             value_name="label").rename(columns={"recording_name": "recording"}))
    rows = pd.concat(rows, ignore_index=True).dropna(subset=["label"])
    return rows.astype({"label": int, "site": str, "recording": str})


def progress():
    """Have and still needed per class, counted at the caps, most needed first."""
    rows = class_rows()
    pos = rows[rows.label == 1]
    clips = (pos.groupby(["cls", "site", "recording"]).size().clip(upper=MAX_PER_RECORDING)
             .groupby(["cls", "site"]).sum().clip(upper=MAX_PER_SITE))
    neg = rows[rows.label == 0].merge(pos[["cls", "site"]].drop_duplicates())  # absent, at a site where it occurs
    have = pd.DataFrame({
        "clips": clips.groupby("cls").sum(),
        "sites": pos.groupby("cls").site.nunique(),
        "recordings": pos.groupby("cls").recording.nunique(),
        "negatives": neg.groupby(["cls", "site"]).size().clip(upper=MAX_NEG_PER_SITE).groupby("cls").sum(),
    }).reindex(CLASSES).fillna(0).astype(int)
    need = pd.DataFrame({f"+ {k}": (t - have[k]).clip(lower=0) for k, t in TARGETS.items()})
    return pd.concat([need, have], axis=1).sort_values(["+ clips", "+ sites"], ascending=False)

## 1. Choose a class

Pick the class you're collecting. The map in step 2 then only shows sites that don't have it yet:

- not in `data/metadata_noise_model.csv`
- not in your labels
- not sampled for it before

You still label every class you hear in step 4, so the other classes build up as by-products.

Site names have to match exactly: a site only drops off the map if the API spells it the way
`metadata_noise_model.csv` does.

In [3]:
CLASS = "power_tools"  # one of CLASSES

In [4]:
assert CLASS in CLASSES, f"CLASS must be one of: {', '.join(CLASSES)}"
sampled = pd.read_csv(CLIPS) if CLIPS.exists() else pd.DataFrame(columns=CLIP_COLUMNS)
rows = class_rows()
acquired = set(rows.site[(rows.cls == CLASS) & (rows.label == 1)]) | set(sampled.site[sampled.detected_label == CLASS])
available = sites[~sites.site.isin(acquired)].reset_index(drop=True)
print(f"{CLASS}: {len(acquired)} sites acquired, {len(acquired & set(sites.site))} of them on the API. "
      f"The map shows the other {len(available)} of the API's {len(sites)} sites.")
progress().loc[[CLASS]]

power_tools: 16 sites acquired, 15 of them on the API. The map shows the other 567 of the API's 582 sites.


,+ clips,+ sites,+ recordings,+ negatives,clips,sites,recordings,negatives
cls,,,,,,,,
power_tools,167,4,19,65,33,16,21,35


## 2. Pick sites

Click a site to select it (it turns red), and click it again to unselect it. Box or lasso select also works. Hover
over a site to see its name and number of recordings.

In [5]:
class SiteMap(go.FigureWidget):
    # Panning or zooming sends a "map._derived" update that plotly reports as an error. Drop it.
    def plotly_relayout(self, relayout_data, **kwargs):
        super().plotly_relayout({k: v for k, v in relayout_data.items() if not k.endswith("._derived")}, **kwargs)


fig = SiteMap(px.scatter_map(
    available, lat="lat", lon="lon", hover_name="site", hover_data=["recordings"],
    center=dict(lat=available.lat.mean(), lon=available.lon.mean()), zoom=5, height=600,
))
fig.update_traces(marker_size=10, selected_marker_color="crimson")
fig.update_layout(clickmode="event", margin=dict(l=0, r=0, t=0, b=0))
picked = set()  # rows of `available` selected on the map


def select_area(trace, points, state):
    """Box or lasso: select exactly the sites inside it."""
    global picked
    picked = set(points.point_inds)
    trace.selectedpoints = sorted(picked) or None


def toggle_site(trace, points, state):
    """Click: select or unselect one site."""
    picked.symmetric_difference_update(points.point_inds)
    trace.selectedpoints = sorted(picked) or None  # None = nothing selected; [] would grey out every site


fig.data[0].on_selection(select_area)
fig.data[0].on_click(toggle_site)
fig

SiteMap({
    'data': [{'customdata': {'bdata': ('LABAAEAAQAAuAC4AEAAOAC4AHgAtAC' ... 'AwAC4AEAAQABAAEAAQABAAEAAeAB4A'),
                             'dtype': 'i2',
                             'shape': '567, 1'},
              'hovertemplate': ('<b>%{hovertext}</b><br><br>lat' ... '{customdata[0]}<extra></extra>'),
              'hovertext': array(['ACWA1', 'ACWA2', 'ACWA3', ..., 'ZASL4', 'ZAY1', 'ZAY3'],
                                 shape=(567,), dtype=object),
              'lat': {'bdata': ('kTAryQ6TRECUWbVSHZlEQPnj7lp3lk' ... '1c2jaNQkDV8oIHhYZCQHZEcu6EjUJA'),
                      'dtype': 'f8'},
              'legendgroup': '',
              'lon': {'bdata': ('eQ0ZRHhEXsCeDI6Sl0NewAfQ7/tHRF' ... 'udWqKCXsAtiYxCIYRewFK5lLSSgV7A'),
                      'dtype': 'f8'},
              'marker': {'color': '#636efa', 'size': 10},
              'mode': 'markers',
              'name': '',
              'selected': {'marker': {'color': 'crimson'}},
              'showlegend': False,

## 3. Sample clips

Finds the `CLASS` detections at the sites you picked and samples them with the same limits as
`data/metadata_noise_model.csv`:

- **Recordings spread over each site's dates:** they're split into equal groups by date and one is taken from each,
  as in notebook 03, using 2 or more recordings when the site has them.
- **At most 5 clips per recording**, at least 60 s apart so each clip is a separate event.
- **At most 10 clips per site.**
- **`HOURS` and `MONTHS`** only apply to recordings with a date and time in the file name.

New clips are added to `data/annotation/noise_clips.csv`. Sites already sampled for `CLASS` are skipped, so
rerunning never adds clips twice. The same sites and settings always pick the same clips.

In [9]:
def norm(name):
    """A species name as a class name: 'Power tools' -> 'power_tools', 'Human non-vocal' -> 'human_non_vocal'."""
    return re.sub(r"[\s_-]+", "_", str(name).strip().lower())


def find_detections(deployment_id):
    """Detections of CLASS at one deployment (a recorder at a site) at or above MIN_CONFIDENCE."""
    species = query_api("/detections/", deployment=deployment_id, confidence=MIN_CONFIDENCE, count=True)
    names = [s["common_name"] for s in species if norm(s["common_name"]) == CLASS]
    rows = [d for name in names
            for d in query_api("/detections/", deployment=deployment_id, confidence=MIN_CONFIDENCE, common_name=name)]
    return pd.DataFrame(rows, columns=["recording_id", "common_name", "confidence", "start_time", "end_time"])


def spread(items, k, rng):
    """k items spread over an ordered list: split it into k equal groups and pick one at random from each."""
    return [items[rng.choice(group)] for group in np.array_split(np.arange(len(items)), k)]


def far_apart(dets, taken, n, rng):
    """Up to n detections in random order, each >= MIN_GAP_SECONDS from the others and from `taken` (start times)."""
    starts, rows = list(taken), []
    for i in rng.permutation(len(dets)):
        if len(rows) == n:
            break
        t = dets.start_time.iloc[i]
        if all(abs(t - s) >= MIN_GAP_SECONDS for s in starts):
            starts.append(t)
            rows.append(i)
    return dets.iloc[rows]


def sample_site(dets, taken, rng):
    """Up to MAX_PER_SITE clips at one site: recordings spread over its dates, <= MAX_PER_RECORDING clips each.
    `taken`: start times of clips already in the clip list, per recording."""
    recs = dets.sort_values(["time", "recording_id"]).recording_id.unique().tolist()
    picks, n = [], 0
    while recs and n < MAX_PER_SITE:
        k = min(len(recs), -(-(MAX_PER_SITE - n) // MAX_PER_RECORDING))  # recordings still needed
        for rec in spread(recs, k, rng):
            clips = far_apart(dets[dets.recording_id == rec], taken.get(rec, []),
                              min(MAX_PER_RECORDING, MAX_PER_SITE - n), rng)
            picks.append(clips)
            n += len(clips)
            recs.remove(rec)
    return pd.concat(picks)

In [10]:
MIN_CONFIDENCE = 0.6  # 0 to 1: how sure the detector must be
HOURS = None          # hours (as in the file names) to sample, e.g. [6, 7, 8, 9]; None = all
MONTHS = None         # months to sample, e.g. [6, 7]; None = all

In [11]:
sampled = pd.read_csv(CLIPS) if CLIPS.exists() else pd.DataFrame(columns=CLIP_COLUMNS)
done = set(sampled.site[sampled.detected_label == CLASS])
selected = [s for s in available.site.iloc[sorted(picked)] if s not in done]
assert selected, f"Select at least one site on the map above (sites already sampled for {CLASS} are skipped)"
print("Sites:", ", ".join(selected))

deployment_site = recordings[recordings.site.isin(selected)].groupby("deployment_id").site.first()
with ThreadPoolExecutor(max_workers=8) as pool:  # each API call takes 5-20 s, so run them in parallel
    hits = pd.concat(pool.map(find_detections, deployment_site.index), ignore_index=True)
assert len(hits), f"No {CLASS} detections at these sites at confidence >= {MIN_CONFIDENCE}"
hits["recording_id"] = hits.recording_id.astype(int)  # pd.concat can leave it as float
hits = hits.merge(recordings[["id", "site", "path", "time", "recording_name"]].rename(columns={"id": "recording_id"}),
                  on="recording_id").drop_duplicates(["recording_id", "start_time"])
when = hits.time + pd.to_timedelta(hits.start_time, unit="s")
keep = pd.Series(True, index=hits.index)
if HOURS:
    keep &= when.dt.hour.isin(HOURS) | when.isna()
if MONTHS:
    keep &= when.dt.month.isin(MONTHS) | when.isna()
hits = hits[keep]

taken = sampled.groupby("recording_id").start_time.agg(list).to_dict()  # clips already in the list, any class
new = pd.concat([sample_site(dets, taken, np.random.default_rng([SEED, zlib.crc32(f"{CLASS}|{site}".encode())]))
                 for site, dets in hits.groupby("site")])  # seeded per site: the same picks whatever else is selected
new = new.assign(id=new.recording_id.astype(str) + "_" + new.start_time.astype(str), detected_label=CLASS,
                 path="s3://" + new.path, sampled_on=pd.Timestamp.today().date().isoformat())[CLIP_COLUMNS]
CLIPS.parent.mkdir(parents=True, exist_ok=True)
(pd.concat([sampled, new]) if len(sampled) else new).drop_duplicates("id").to_csv(CLIPS, index=False)

print(f"Added {len(new)} clips from {new.recording_id.nunique()} recordings at {new.site.nunique()} sites to {CLIPS}")
for site in sorted(set(selected) - set(new.site)):
    print(f"No {CLASS} detections at {site}")
new.groupby("site").agg(recordings=("recording_id", "nunique"), clips=("id", "size"))

Sites: CBER3, CCWA4, CDLOER3, CLFE1, CRER4, CSVER4, EDSA1, GRHO3, KELS4b, MOST2, RACR3, SFSB4, STER4


AssertionError: No power_tools detections at these sites at confidence >= 0.6

## 4. Label

For each clip:

- **Press every class you hear** in the 3-second clip, not just the detected one. Pressed = 1, the rest = 0.
- **Submit** saves the clip and moves on. Submitting with nothing pressed saves it as a negative for all 13 classes,
  which is still useful.
- **Skip** moves on without saving, so the clip doesn't count.
- **unsure** or **unusable** flags a clip, and flagged clips don't count.

**Keys:** click any button once, or Tab to the button grid. Focus then stays there from clip to clip:

| Key | Does |
|---|---|
| `1`–`9`, `0`, `q`, `w`, `e` | press the classes, in order |
| `u` / `x` | flag as unsure / unusable |
| `s` | save, the same as **Submit** |
| `k` | skip (doesn't save) |
| `j` | go back to the previous clip |

Labels save to `data/annotation/noise_labels.csv` as you go. The form is set up in `annotator_config/`.

In [ ]:
ann = BioacousticAnnotator(project="annotator_config/projects/noise.yaml")
for msg in ann.validate()["errors"]:
    print("❌", msg)
ann.open()

## 5. Progress

How far each class is from its target: `data/metadata_noise_model.csv` plus every clip labeled so far, counted with
the same caps (at most 5 per recording and 10 per site). **Negatives** are labeled clips without the class, at sites
where it occurs, with up to 5 per site counted. Most needed first. Rerun any time.

In [ ]:
progress()